Импорты, seed и среда



In [ ]:
import random
from typing import Iterable, List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline
)
from transformers.utils.notebook import NotebookProgressCallback


from datasets import load_dataset, Dataset

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Данные и первичный анализ

In [ ]:
dataset = load_dataset("emotion")

print(f"Классы: {dataset['train'].features['label'].names}")
print(f"Размер train: {len(dataset['train'])}, validation: {len(dataset['validation'])}, test: {len(dataset['test'])}")
print(dataset)

In [ ]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

df = pd.concat([train_df, val_df, test_df])
label_names = dataset['train'].features['label'].names
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}

df["label_id"] = df["label"].map(label2id)

print("label2id:", label2id)
print("id2label:", id2label)

display(df.sample(6, random_state=42).reset_index(drop=True))

display("Распределение во всём датасете:", df["label"].value_counts())
display("Train-распределение:", train_df["label"].value_counts())
display("Validation-распределение:" ,val_df["label"].value_counts())
display("Test-распределение:", test_df["label"].value_counts())

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
print("Размер датасета:", len(dataset))
train_data = dataset['train']

print("Размер тренировочного датасета:", len(train_data))

display(pd.Series(train_data["label"]).value_counts())

label_names = train_data.features['label'].names

label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}

print(f"Классы: {label_names}")
print(f"Пример маппинга: {id2label}")

In [ ]:
label_names = dataset['train'].features['label'].names
for i in range(5):
    item = dataset['train'][i]
    text = item['text']
    label_id = item['label']
    label_name = label_names[label_id]
    print(f"[{label_name.upper():8s}] : \"{text}\"")

In [ ]:
# Компактная русскоязычная BERT-подобная модель.
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texts = [
    dataset["train"][0]["text"],
    dataset["train"][1]["text"],
    dataset["train"][4]["text"],
    dataset["train"][6]["text"]
]


print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)

def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )

tokenized_datasets = dataset.map(tokenize_batch, batched=True)

# Удаляем столбец "text": DataCollatorWithPadding попытается преобразовать
# все поля датасета в тензоры, а строки в тензор не конвертируются.
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets

In [ ]:
# Data collator будет добавлять padding динамически, прямо при формировании батча.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_batch = [tokenized_datasets["train"][i] for i in range(3)]
collated_batch = data_collator(sample_batch)

for key, value in collated_batch.items():
    print(f"{key}: shape={tuple(value.shape)}")

In [ ]:
batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    add_special_tokens=True,
    return_tensors="pt",
)

num_samples = batch["input_ids"].shape[0]

data = []

for i in range(num_samples):
    original_text = texts[i]

    ids_list = batch["input_ids"][i].tolist()
    mask_list = batch["attention_mask"][i].tolist()

    tokens = tokenizer.convert_ids_to_tokens(ids_list)

    tokens_display = " ".join(tokens)
    ids_display = str(ids_list)
    mask_display = str(mask_list)

    print(f"Пример {i+1}")
    print(f"{original_text:} | {tokens_display:} | input_ids: {ids_display}")
    print(f"{'':25}   {'':45}   attention_mask: {mask_display}\n")

In [ ]:
text_clf = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    tokenizer=tokenizer,
    device=device,
)

print("Классы датасета:\n", label_names)

Была использовована модель j-hartmann/emotion-english-distilroberta-base. Она уже дообучена (fine-tuned) на задачу классификации эмоций. Это значит, что её последний слой настроен выдавать вероятности именно для классов sadness, joy, love, anger, fear, surprise, поэтому она сразу выдаёт осмысленный результат. Но последний результат спорный, так как тут нужно смотреть на то, как он интерпретирован

In [ ]:
emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
emotion_classifier = pipeline("text-classification", model=emotion_model_name, top_k=None)

test_texts = [
    "I am so happy and excited!",
    "This is terrible and I am angry.",
    "I feel sad and lonely today.",
    "Wow, I didn't expect that at all!",
    "You are always eatting in class!"
]

results = []
for text in test_texts:
    preds = emotion_classifier(text)[0]
    top_pred = max(preds, key=lambda x: x['score'])
    results.append({
        "Text": text,
        "Predicted Emotion": top_pred['label'],
        "Score": f"{top_pred['score']:.2%}"
    })

df_results = pd.DataFrame(results)
display(df_results)

In [ ]:
# Вспомогательная функция для ручного инференса одного текста.
def predict_one_text(text: str) -> Dict[str, Any]:
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)

    pred_id = int(torch.argmax(probs, dim=-1).item())
    pred_label = model.config.id2label[pred_id]
    pred_score = float(probs[0, pred_id].item())

    result = {
        "text": text,
        "pred_id": pred_id,
        "pred_label": pred_label,
        "pred_score": pred_score,
        "logits": logits.cpu().numpy().round(4).tolist()[0],
        "probabilities": probs.cpu().numpy().round(4).tolist()[0],
    }
    return result



# Удобная функция для преобразования вероятностей в таблицу.
def probability_table(text: str) -> pd.DataFrame:
    result = predict_one_text(text)
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    df = pd.DataFrame(
        {
            "label": labels,
            "probability": result["probabilities"],
        }
    ).sort_values("probability", ascending=False, ignore_index=True)
    df.insert(0, "text", text)
    return df

In [ ]:
label_names = dataset['train'].features['label'].names
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}


# Загружаем модель для классификации.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:
# Функция метрик для Trainer.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

In [ ]:
# Общие параметры обучения.
common_training_kwargs = dict(
    output_dir="outputs/hw13_aiecity_bert_finetuning_demo",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=2,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

try:
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )
except TypeError:
    training_args = TrainingArguments(
        eval_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )

training_args

In [ ]:
# Собираем Trainer и запускаем обучение.
# В transformers >= 5.0 аргумент tokenizer переименован в processing_class.
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

train_result = trainer.train()
train_result

Оценка качества и краткий анализ ошибок

In [ ]:
# История логов Trainer.
history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.head(10))

plt.figure(figsize=(8, 4))

if "loss" in history_df.columns:
    train_logs = history_df.dropna(subset=["loss"])
    plt.plot(train_logs["step"], train_logs["loss"], marker="o", label="train loss")

if "eval_loss" in history_df.columns:
    eval_logs = history_df.dropna(subset=["eval_loss"])
    plt.plot(eval_logs["step"], eval_logs["eval_loss"], marker="s", label="eval loss")

plt.title("История обучения")
plt.xlabel("Шаг")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("./artifacts/training_curves.png")
plt.show()

In [ ]:
# В transformers >= 5.x NotebookProgressCallback теряет состояние после обучения,
# что вызывает RuntimeError при вызове evaluate() вне тренировочного цикла.
# Удаляем его перед standalone-оценкой – это стандартный обходной путь.
trainer.remove_callback(NotebookProgressCallback)

# Оценка Trainer на validation и test.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

In [ ]:
# Детальные предсказания на тестовой выборке.
test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_true = test_output.label_ids

print("Classification report on test:")
print(
    classification_report(
        test_true,
        test_preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        zero_division=0,
    )
)

cm = confusion_matrix(test_true, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=30, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("./artifacts/confusion_matrix.png")
plt.show()

In [ ]:
# Таблица ошибок на тестовой выборке.
test_texts = test_df.reset_index(drop=True)["text"]
error_rows = []

for text, true_id, pred_id, prob_vector in zip(test_texts, test_true, test_preds, torch.softmax(torch.tensor(test_logits), dim=-1).numpy()):
    if true_id != pred_id:
        error_rows.append({
            "text": text,
            "true_label": id2label[int(true_id)],
            "pred_label": id2label[int(pred_id)],
            "pred_confidence": float(prob_vector[pred_id]),
        })

errors_df = pd.DataFrame(error_rows)

if len(errors_df) == 0:
    print("На тестовой выборке ошибок не найдено. Это возможно на маленьком учебном датасете.")
else:
    display(errors_df.sort_values(by="pred_confidence", ascending=False).reset_index(drop=True))

errors_df.to_csv("./artifacts/sample_predictions.csv")

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,
    random_state=42,
    stratify=train_df["label"],
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nРаспределение классов в train:")
display(train_df["label"].value_counts())

print("Распределение классов в validation:")
display(val_df["label"].value_counts())

print("Распределение классов в test:")
display(test_df["label"].value_counts())

In [ ]:
my_texts = [
    "I am so happy and excited about this!",
    "This is terrible and I am angry.",
    "I feel sad and lonely today.",
    "Etot tekst demonstriruet razbienie na tokeny",
    "Wow, I didn't expect that at all!",
    "The weather is nice today.",
    "Khorosho zhit!",
]

try:
    classifier = emotion_classifier
except NameError:
    classifier = None

with open("tokenization_examples.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("ПРИМЕРЫ ТОКЕНИЗАЦИИ (Ваши собственные данные)\n")
    f.write("="*80 + "\n\n")

    for i, txt in enumerate(my_texts):
        if classifier:
            preds = classifier(txt)[0]
            top_pred = max(preds, key=lambda x: x['score'])
            label_name = top_pred['label'].upper()
        else:
            label_name = "UNKNOWN"

        # Токенизация
        encoded = tokenizer(txt, padding='max_length', truncation=True, max_length=20, return_tensors='pt')

        ids = encoded['input_ids'][0].tolist()
        mask = encoded['attention_mask'][0].tolist()
        tokens = tokenizer.convert_ids_to_tokens(ids)

        # Запись в файл
        f.write(f"--- Пример {i+1} (Предсказанный класс: {label_name}) ---\n")
        f.write(f"Original Text: \"{txt}\"\n")
        f.write("-"*80 + "\n")
        f.write(f"Tokens      : {' '.join(tokens)}\n")
        f.write(f"Input IDs   : {ids}\n")
        f.write(f"Attention Mask: {mask}\n")
        f.write("\n")